# Transformación de Datos - Práctica Semana 2

## Inteligencia de Negocios

## Grupo 9

### Integrantes

* Guevara Paz y Miño Juan Diego
* Moreno Jeremy
* Solis Paulo

---

## Descripción del proyecto

El presente proyecto se desarrolla en el contexto de un proceso ETL aplicado al análisis de datos relacionados con ciberseguridad global e indicadores digitales por país. Para ello, se utilizan tres fuentes de datos principales: incidentes globales de ciberseguridad, índices de ciberseguridad por país y estadísticas de usuarios de internet.

El objetivo del proyecto es preparar los datos para su posterior análisis mediante procesos de limpieza, transformación, normalización y validación. Estas actividades permiten mejorar la calidad de la información, reducir inconsistencias y facilitar la integración entre las fuentes utilizadas.

Durante esta práctica se trabajará principalmente en la etapa de transformación del proceso ETL. Se revisará la estructura de los datos, se identificarán valores nulos, duplicados e inconsistencias, se seleccionarán columnas relevantes, se aplicarán funciones de limpieza y se generarán datos transformados que puedan ser utilizados para análisis posteriores o carga en una base de datos.

El dominio seleccionado, ciberseguridad, permite relacionar información sobre tipos de ataques, pérdidas financieras, usuarios afectados, mecanismos de defensa, indicadores de madurez digital y penetración de internet. Esto aporta una base útil para comprender tendencias, riesgos y oportunidades de mejora en la gestión de la seguridad de la información.

## 1. Exploración inicial de las fuentes de datos

Antes de aplicar procesos de limpieza y transformación, se realiza una exploración inicial de las tres fuentes de datos utilizadas en el proyecto. Esta revisión permite comprender la estructura general de cada archivo, identificar la cantidad de filas y columnas, revisar los tipos de datos, detectar valores nulos, registros duplicados, columnas relevantes y posibles campos de relación entre las fuentes.

Las fuentes analizadas son:

* **Global Cybersecurity Threats 2015-2024:** contiene información sobre incidentes globales de ciberseguridad, incluyendo país, año, tipo de ataque, industria afectada, pérdidas financieras, usuarios afectados, fuente del ataque, vulnerabilidad explotada, mecanismo de defensa y tiempo de resolución.
* **Cyber Security Index:** contiene indicadores de ciberseguridad por país, como CEI, GCI, NCSI y DDL.
* **Internet Users by Country:** contiene información sobre usuarios de internet, población, región, subregión, año y porcentaje de usuarios de internet por país.

Esta fase es importante porque permite identificar problemas de calidad antes de transformar los datos. De esta forma, se pueden tomar decisiones adecuadas sobre limpieza, normalización y selección de columnas para el análisis posterior.

In [54]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Configuración para visualizar mejor los resultados en el notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Ruta base de los archivos CSV
DATA_DIR = Path("data")

# Rutas de las tres fuentes de datos
ruta_threats = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
ruta_cyber_security = DATA_DIR / "Cyber_security.csv"
ruta_internet_users = DATA_DIR / "internet_users_by_country_cleaned.csv"

# Validación básica de existencia de archivos antes de cargarlos
rutas = [ruta_threats, ruta_cyber_security, ruta_internet_users]
archivos_faltantes = [str(ruta) for ruta in rutas if not ruta.exists()]

if archivos_faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos CSV. "
        "Verifica que la carpeta 'data' esté en la misma ubicación del notebook: "
        + ", ".join(archivos_faltantes)
    )

# Carga de las tres fuentes de datos
df_threats = pd.read_csv(ruta_threats)
df_cyber_security = pd.read_csv(ruta_cyber_security)
df_internet_users = pd.read_csv(ruta_internet_users)

print("Fuentes de datos cargadas correctamente.")

Fuentes de datos cargadas correctamente.


In [55]:
# Crear un diccionario para manejar las fuentes de forma organizada
datasets = {
    "Global Cybersecurity Threats": df_threats,
    "Cyber Security Index": df_cyber_security,
    "Internet Users by Country": df_internet_users
}

# Resumen general de filas, columnas, duplicados y valores nulos
resumen_general = []

for nombre, df in datasets.items():
    resumen_general.append({
        "fuente": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "registros_duplicados": int(df.duplicated().sum()),
        "total_valores_nulos": int(df.isnull().sum().sum())
    })

df_resumen_general = pd.DataFrame(resumen_general)
df_resumen_general

,fuente,filas,columnas,registros_duplicados,total_valores_nulos
0,Global Cybersecurity Threats,3000,10,0,0
1,Cyber Security Index,192,6,0,151
2,Internet Users by Country,215,7,0,0


In [56]:
# Revisión de columnas, tipos de datos, valores nulos y valores únicos por cada fuente
for nombre, df in datasets.items():
    print(f"\nFuente: {nombre}")
    display(pd.DataFrame({
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
        "valores_nulos": df.isnull().sum().values,
        "porcentaje_nulos": (df.isnull().mean().values * 100).round(2),
        "valores_unicos": df.nunique().values
    }))


Fuente: Global Cybersecurity Threats


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country,str,0,0.0,10
1,Year,int64,0,0.0,10
2,Attack Type,str,0,0.0,6
3,Target Industry,str,0,0.0,7
4,Financial Loss (in Million $),float64,0,0.0,2536
5,Number of Affected Users,int64,0,0.0,2998
6,Attack Source,str,0,0.0,4
7,Security Vulnerability Type,str,0,0.0,4
8,Defense Mechanism Used,str,0,0.0,5
9,Incident Resolution Time (in Hours),int64,0,0.0,72



Fuente: Cyber Security Index


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country,str,0,0.00,192
1,Region,str,0,0.00,5
2,CEI,float64,84,43.75,85
3,GCI,float64,2,1.04,180
4,NCSI,float64,25,13.02,69
5,DDL,float64,40,20.83,150



Fuente: Internet Users by Country


,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,Country or area,str,0,0.0,215
1,Subregion,str,0,0.0,22
2,Region,str,0,0.0,5
3,Internet users,int64,0,0.0,215
4,Population_2021,int64,0,0.0,215
5,Year,int64,0,0.0,5
6,Percentage_Internet_Users,float64,0,0.0,195


In [57]:
# Mostrar las primeras filas de cada fuente para revisar su estructura visualmente
for nombre, df in datasets.items():
    print(f"\nVista preliminar: {nombre}")
    display(df.head())


Vista preliminar: Global Cybersecurity Threats


,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68



Vista preliminar: Cyber Security Index


,Country,Region,CEI,GCI,NCSI,DDL
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50
1,Albania,Europe,0.566,64.32,62.34,48.74
2,Algeria,Africa,0.721,33.95,33.77,42.81
3,Andorra,Europe,NaN,26.38,NaN,NaN
4,Angola,Africa,NaN,12.99,9.09,22.69



Vista preliminar: Internet Users by Country


,Country or area,Subregion,Region,Internet users,Population_2021,Year,Percentage_Internet_Users
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6
2,United States,Northern America,Americas,311300000,336997624,2023,92.4
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8


In [58]:
# Revisión específica de registros duplicados
duplicados = []

for nombre, df in datasets.items():
    duplicados.append({
        "fuente": nombre,
        "cantidad_duplicados": int(df.duplicated().sum())
    })

df_duplicados = pd.DataFrame(duplicados)
df_duplicados

,fuente,cantidad_duplicados
0,Global Cybersecurity Threats,0
1,Cyber Security Index,0
2,Internet Users by Country,0


In [59]:
from pandas.api.types import is_object_dtype, is_string_dtype

def revisar_columnas_texto(df, nombre_fuente):
    """
    Revisa columnas de tipo texto para identificar posibles problemas de formato,
    como espacios al inicio o final, textos vacíos y cantidad de valores únicos.
    """

    # Seleccionar columnas de texto de forma compatible con versiones actuales y futuras de pandas
    columnas_texto = [
        columna for columna in df.columns
        if is_object_dtype(df[columna]) or is_string_dtype(df[columna])
    ]

    resultados = []

    for columna in columnas_texto:
        serie = df[columna].astype("string")

        resultados.append({
            "fuente": nombre_fuente,
            "columna": columna,
            "valores_unicos": int(df[columna].nunique()),
            "valores_con_espacios": int((serie != serie.str.strip()).sum()),
            "valores_vacios": int(serie.str.strip().eq("").sum())
        })

    return pd.DataFrame(resultados)


revision_texto = pd.concat(
    [revisar_columnas_texto(df, nombre) for nombre, df in datasets.items()],
    ignore_index=True
)

revision_texto

,fuente,columna,valores_unicos,valores_con_espacios,valores_vacios
0,Global Cybersecurity Threats,Country,10,0,0
1,Global Cybersecurity Threats,Attack Type,6,0,0
2,Global Cybersecurity Threats,Target Industry,7,0,0
3,Global Cybersecurity Threats,Attack Source,4,0,0
4,Global Cybersecurity Threats,Security Vulnerability Type,4,0,0
5,Global Cybersecurity Threats,Defense Mechanism Used,5,0,0
6,Cyber Security Index,Country,192,0,0
7,Cyber Security Index,Region,5,0,0
8,Internet Users by Country,Country or area,215,0,0
9,Internet Users by Country,Subregion,22,0,0


In [60]:
# Revisión de columnas que podrían servir para relacionar las fuentes
columnas_relacion = pd.DataFrame({
    "campo": ["Country / Country or area", "Year", "Region"],
    "presente_en": [
        "Global Cybersecurity Threats, Cyber Security Index, Internet Users by Country",
        "Global Cybersecurity Threats, Internet Users by Country",
        "Cyber Security Index, Internet Users by Country"
    ],
    "uso_potencial": [
        "Permite relacionar las tres fuentes a nivel de país, previa normalización de nombres.",
        "Permite analizar datos por año, aunque debe validarse porque no todas las fuentes manejan el mismo rango temporal.",
        "Permite realizar análisis regionales, siempre que los nombres de regiones estén homologados."
    ]
})

columnas_relacion

,campo,presente_en,uso_potencial
0,Country / Country or area,"Global Cybersecurity Threats, Cyber Security I...",Permite relacionar las tres fuentes a nivel de...
1,Year,"Global Cybersecurity Threats, Internet Users b...","Permite analizar datos por año, aunque debe va..."
2,Region,"Cyber Security Index, Internet Users by Country","Permite realizar análisis regionales, siempre ..."


In [61]:
# Revisión de países por fuente
print("Cantidad de países en Global Cybersecurity Threats:", df_threats["Country"].nunique())
print("Cantidad de países en Cyber Security Index:", df_cyber_security["Country"].nunique())
print("Cantidad de países en Internet Users by Country:", df_internet_users["Country or area"].nunique())

# Países presentes en el dataset de amenazas globales
paises_threats = sorted(df_threats["Country"].unique())

print("\nPaíses presentes en Global Cybersecurity Threats:")
for pais in paises_threats:
    print("-", pais)

# Revisión de regiones en las fuentes que tienen columna Region
print("\nRegiones en Cyber Security Index:")
display(df_cyber_security["Region"].value_counts().rename_axis("region").reset_index(name="cantidad"))

print("\nRegiones en Internet Users by Country:")
display(df_internet_users["Region"].value_counts().rename_axis("region").reset_index(name="cantidad"))

Cantidad de países en Global Cybersecurity Threats: 10
Cantidad de países en Cyber Security Index: 192
Cantidad de países en Internet Users by Country: 215

Países presentes en Global Cybersecurity Threats:
- Australia
- Brazil
- China
- France
- Germany
- India
- Japan
- Russia
- UK
- USA

Regiones en Cyber Security Index:


,region,cantidad
0,Asia-Pasific,56
1,Africa,53
2,Europe,48
3,North America,24
4,South America,11



Regiones en Internet Users by Country:


,region,cantidad
0,Africa,55
1,Asia,49
2,Europe,47
3,Americas,45
4,Oceania,19


In [62]:
# Homologación preliminar de nombres de países para comparación
country_mapping = {
    "USA": "United States",
    "UK": "United Kingdom"
}

paises_threats_normalizados = set(
    df_threats["Country"]
    .replace(country_mapping)
    .str.strip()
)

paises_cyber_security = set(
    df_cyber_security["Country"]
    .str.strip()
)

paises_internet_users = set(
    df_internet_users["Country or area"]
    .str.strip()
)

# Validar coincidencias entre fuentes
coinciden_cyber_security = paises_threats_normalizados.intersection(paises_cyber_security)
coinciden_internet_users = paises_threats_normalizados.intersection(paises_internet_users)
coinciden_todas = paises_threats_normalizados.intersection(paises_cyber_security).intersection(paises_internet_users)

print("Países del dataset de amenazas que coinciden con Cyber Security Index:", len(coinciden_cyber_security))
print(sorted(coinciden_cyber_security))

print("\nPaíses del dataset de amenazas que coinciden con Internet Users:", len(coinciden_internet_users))
print(sorted(coinciden_internet_users))

print("\nPaíses que coinciden en las tres fuentes:", len(coinciden_todas))
print(sorted(coinciden_todas))

Países del dataset de amenazas que coinciden con Cyber Security Index: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']

Países del dataset de amenazas que coinciden con Internet Users: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']

Países que coinciden en las tres fuentes: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']


In [63]:
# Selección preliminar de columnas relevantes para el proyecto
columnas_relevantes = {
    "Global Cybersecurity Threats": [
        "Country",
        "Year",
        "Attack Type",
        "Target Industry",
        "Financial Loss (in Million $)",
        "Number of Affected Users",
        "Attack Source",
        "Security Vulnerability Type",
        "Defense Mechanism Used",
        "Incident Resolution Time (in Hours)"
    ],
    "Cyber Security Index": [
        "Country",
        "Region",
        "CEI",
        "GCI",
        "NCSI",
        "DDL"
    ],
    "Internet Users by Country": [
        "Country or area",
        "Subregion",
        "Region",
        "Internet users",
        "Population_2021",
        "Year",
        "Percentage_Internet_Users"
    ]
}

for fuente, columnas in columnas_relevantes.items():
    print(f"\nColumnas relevantes para {fuente}:")
    for columna in columnas:
        print(f"- {columna}")


Columnas relevantes para Global Cybersecurity Threats:
- Country
- Year
- Attack Type
- Target Industry
- Financial Loss (in Million $)
- Number of Affected Users
- Attack Source
- Security Vulnerability Type
- Defense Mechanism Used
- Incident Resolution Time (in Hours)

Columnas relevantes para Cyber Security Index:
- Country
- Region
- CEI
- GCI
- NCSI
- DDL

Columnas relevantes para Internet Users by Country:
- Country or area
- Subregion
- Region
- Internet users
- Population_2021
- Year
- Percentage_Internet_Users


### Hallazgos de la exploración inicial

Después de revisar las tres fuentes de datos, se identificaron los siguientes hallazgos:

#### Global Cybersecurity Threats 2015-2024

Esta fuente contiene información detallada sobre incidentes globales de ciberseguridad. Está compuesta por registros asociados a países, años, tipos de ataque, industrias afectadas, pérdidas financieras, usuarios afectados, vulnerabilidades, mecanismos de defensa y tiempo de resolución.

Durante la exploración inicial se observa que esta fuente cuenta con 3000 registros y 10 columnas. No presenta valores nulos en sus columnas principales y no contiene registros duplicados. Sus columnas son relevantes para el proyecto porque permiten analizar el comportamiento de los ataques, el impacto económico, la cantidad de usuarios afectados y los tiempos de respuesta ante incidentes.

La columna **Country** puede utilizarse como campo de relación con las otras fuentes, siempre que previamente se normalicen los nombres de países.

#### Cyber Security Index

Esta fuente contiene indicadores de ciberseguridad por país, tales como CEI, GCI, NCSI y DDL. Estos indicadores permiten complementar el análisis de incidentes con variables relacionadas con exposición, madurez y desarrollo digital.

En esta fuente se identifican 192 registros y 6 columnas. También se observan valores nulos en algunas columnas numéricas, principalmente en los indicadores CEI, GCI, NCSI y DDL. Estos valores deberán ser tratados en la fase de limpieza, ya sea mediante imputación, marcado de datos incompletos o conservación controlada según el objetivo del análisis.

También se observa que la columna **Country** puede funcionar como clave de relación con las otras fuentes. Además, la columna **Region** puede ser útil para análisis agrupados por zona geográfica.

#### Internet Users by Country

Esta fuente contiene datos sobre usuarios de internet, población, región, subregión, año y porcentaje de usuarios de internet por país. Su información es importante porque permite contextualizar los incidentes de ciberseguridad en función del nivel de conectividad digital de cada país.

Durante la exploración inicial se identifican 215 registros y 7 columnas. No se evidencian valores nulos ni registros duplicados. Las columnas más relevantes son **Country or area**, **Internet users**, **Population_2021**, **Year** y **Percentage_Internet_Users**.

La columna **Country or area** deberá ser normalizada para poder relacionarse correctamente con la columna **Country** de las otras fuentes. También se debe revisar la columna **Year**, ya que puede ayudar en ciertos análisis temporales, pero no todas las fuentes manejan el mismo rango de años.

#### Posibles columnas de relación

La principal columna de relación entre las tres fuentes es el país. En el dataset de amenazas aparece como **Country**, en el índice de ciberseguridad también como **Country**, y en el dataset de usuarios de internet como **Country or area**.

También existen campos como **Year** y **Region**, pero deben utilizarse con cuidado. El campo **Year** no está presente en todas las fuentes, y las regiones no manejan exactamente las mismas categorías entre datasets.

#### Hallazgo sobre países y regiones

Al revisar la cantidad de países presentes en cada fuente, se observa que el dataset **Global Cybersecurity Threats** contiene únicamente 10 países, mientras que **Cyber Security Index** contiene 192 países y **Internet Users by Country** contiene 215 países o áreas geográficas.

Esta diferencia no representa un error, sino una característica de las fuentes utilizadas. El dataset de amenazas está enfocado en incidentes registrados para un conjunto reducido de países, mientras que las otras dos fuentes tienen una cobertura global más amplia.

Para poder relacionar las tres fuentes, se identifica que la columna principal de unión será el país. Sin embargo, antes de realizar cualquier integración es necesario homologar ciertos nombres, por ejemplo **USA** a **United States** y **UK** a **United Kingdom**.

Después de aplicar esta homologación preliminar, los 10 países presentes en el dataset de amenazas pueden relacionarse con las otras dos fuentes. Esto confirma que es posible integrar los datasets usando el país como campo principal de relación.

También se observa que las regiones no están escritas exactamente igual entre las fuentes. Por ejemplo, en **Cyber Security Index** aparece la región **Asia-Pasific**, mientras que en **Internet Users by Country** se manejan regiones como **Asia**, **Africa**, **Europe**, **Americas** y **Oceania**. Por esta razón, las regiones deberán revisarse y normalizarse si se desean utilizar para análisis comparativos.

En conclusión, las tres fuentes son útiles para el proyecto, pero requieren procesos de limpieza y normalización antes de integrarse o utilizarse en análisis posteriores.


## 2. Configuración segura del proyecto

En esta fase se configura el uso de variables de entorno para proteger información sensible del proyecto, especialmente los parámetros de conexión a PostgreSQL. En lugar de escribir credenciales directamente dentro del notebook, se utiliza un archivo `.env`, el cual permite centralizar la configuración y evitar que datos sensibles queden expuestos en el código.

Las variables de entorno utilizadas en este proyecto son:

- **POSTGRES_USER:** usuario de conexión a PostgreSQL.
- **POSTGRES_PASSWORD:** contraseña del usuario de PostgreSQL.
- **POSTGRES_DB:** nombre de la base de datos utilizada para el proyecto ETL.
- **POSTGRES_HOST:** servidor o host donde se encuentra PostgreSQL.
- **POSTGRES_PORT:** puerto de conexión al servicio de PostgreSQL.

El uso de variables de entorno es importante porque separa la configuración sensible de la lógica del programa. Esto mejora la seguridad, facilita el mantenimiento del proyecto y permite ejecutar el notebook en diferentes ambientes sin modificar directamente el código fuente.

Además, esta práctica aporta a la seguridad del proyecto porque reduce el riesgo de exponer credenciales en repositorios, capturas de pantalla, entregas académicas o archivos compartidos. En un entorno profesional, este enfoque es especialmente relevante para proteger accesos a bases de datos, servicios externos, tokens, rutas privadas o parámetros críticos de infraestructura.

In [64]:
from dotenv import load_dotenv
import os

# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Lectura de variables de conexión a PostgreSQL
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")

# Validar que las variables requeridas existan
variables_requeridas = {
    "POSTGRES_USER": DB_USER,
    "POSTGRES_PASSWORD": DB_PASSWORD,
    "POSTGRES_DB": DB_NAME,
    "POSTGRES_HOST": DB_HOST,
    "POSTGRES_PORT": DB_PORT
}

variables_faltantes = [nombre for nombre, valor in variables_requeridas.items() if not valor]

if variables_faltantes:
    raise EnvironmentError(
        "Faltan variables de entorno requeridas en el archivo .env: "
        + ", ".join(variables_faltantes)
    )

# Mostrar configuración sin exponer información sensible
configuracion_segura = pd.DataFrame({
    "variable": [
        "POSTGRES_USER",
        "POSTGRES_PASSWORD",
        "POSTGRES_DB",
        "POSTGRES_HOST",
        "POSTGRES_PORT"
    ],
    "valor_cargado": [
        DB_USER,
        "********",
        DB_NAME,
        DB_HOST,
        DB_PORT
    ],
    "descripcion": [
        "Usuario de conexión a PostgreSQL",
        "Contraseña protegida y no visible en el notebook",
        "Nombre de la base de datos del proyecto",
        "Host o servidor donde se ejecuta PostgreSQL",
        "Puerto expuesto para la conexión a PostgreSQL"
    ]
})

configuracion_segura

,variable,valor_cargado,descripcion
0,POSTGRES_USER,admin,Usuario de conexión a PostgreSQL
1,POSTGRES_PASSWORD,********,Contraseña protegida y no visible en el notebook
2,POSTGRES_DB,db_ciberseguridad_etl,Nombre de la base de datos del proyecto
3,POSTGRES_HOST,localhost,Host o servidor donde se ejecuta PostgreSQL
4,POSTGRES_PORT,5432,Puerto expuesto para la conexión a PostgreSQL


In [65]:
from sqlalchemy import create_engine, text

# Construcción de la cadena de conexión usando variables de entorno
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Creación del engine de conexión a PostgreSQL
engine = create_engine(connection_string)

try:
    with engine.connect() as connection:
        resultado = connection.execute(text("SELECT current_database(), current_user;"))
        database_name, database_user = resultado.fetchone()

        print("Conexión exitosa a PostgreSQL")
        print("Base de datos conectada:", database_name)
        print("Usuario conectado:", database_user)
except Exception as e:
    print("Error al conectar con PostgreSQL:")
    print(e)

Conexión exitosa a PostgreSQL
Base de datos conectada: db_ciberseguridad_etl
Usuario conectado: admin


### Hallazgos de la configuración segura

La configuración segura del proyecto se realizó mediante el uso de variables de entorno almacenadas en un archivo `.env`. Estas variables permiten manejar los datos de conexión a PostgreSQL sin escribir directamente credenciales sensibles dentro del notebook.

Durante esta fase se cargaron variables como el usuario, contraseña, nombre de la base de datos, host y puerto de conexión. Para evitar exposición de información sensible, la contraseña no se imprime en pantalla y se muestra únicamente como un valor enmascarado.

Esta práctica mejora la seguridad del proyecto porque separa las credenciales de la lógica del código. Además, facilita que el mismo notebook pueda ejecutarse en otros ambientes, cambiando únicamente el archivo `.env` y no el código fuente.

En un contexto profesional, este enfoque es importante porque ayuda a prevenir la filtración accidental de credenciales en repositorios, documentos, entregas, capturas de pantalla o ambientes compartidos. También permite mantener una mejor organización y control sobre los parámetros de conexión utilizados por el proceso ETL.

## 3. Limpieza de datos

En esta fase se aplican funciones de limpieza sobre las tres fuentes del proyecto. El objetivo es preparar los datos para las siguientes etapas del proceso ETL, corrigiendo nombres de columnas, normalizando textos, eliminando duplicados, convirtiendo tipos de datos, tratando valores nulos y limpiando las columnas que podrían utilizarse como claves de relación.

Para mantener el código organizado, se definen funciones reutilizables que permiten aplicar reglas similares a diferentes DataFrames. Esto facilita la trazabilidad del proceso y evita repetir instrucciones de limpieza en varias partes del notebook.

In [66]:
import re
import unicodedata

# Diccionario para homologar nombres de países que aparecen abreviados o con escrituras distintas.
# Esta limpieza es importante porque el país será una de las principales columnas de relación.
COUNTRY_MAPPING = {
    "USA": "United States",
    "U.S.A.": "United States",
    "US": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom"
}


def normalizar_nombre_columna(columna):
    """
    Convierte nombres de columnas a formato snake_case.
    Ejemplo: 'Financial Loss (in Million $)' -> 'financial_loss_in_million'
    """
    columna = columna.strip().lower()
    columna = columna.replace("$", "")
    columna = re.sub(r"[()/%]+", " ", columna)
    columna = re.sub(r"[^a-z0-9]+", "_", columna)
    columna = re.sub(r"_+", "_", columna)
    columna = columna.strip("_")
    return columna


def normalizar_texto(valor):
    """
    Limpia valores de texto eliminando espacios innecesarios.
    Si el valor está vacío o es nulo, devuelve pd.NA.
    """
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)
    
    if valor == "":
        return pd.NA
    
    return valor


def homologar_pais(valor):
    """
    Normaliza y homologa nombres de países para facilitar la relación entre fuentes.
    """
    valor = normalizar_texto(valor)
    
    if pd.isna(valor):
        return pd.NA
    
    return COUNTRY_MAPPING.get(valor, valor)


def limpiar_columnas_texto(df):
    """
    Aplica limpieza básica a todas las columnas de texto de un DataFrame.
    """
    df_limpio = df.copy()
    
    columnas_texto = df_limpio.select_dtypes(include=["object", "string"]).columns
    
    for columna in columnas_texto:
        df_limpio[columna] = df_limpio[columna].apply(normalizar_texto)
    
    return df_limpio


def convertir_columnas_numericas(df, columnas):
    """
    Convierte columnas específicas a valores numéricos.
    Los valores no convertibles se transforman en NaN para ser tratados posteriormente.
    """
    df_limpio = df.copy()
    
    for columna in columnas:
        if columna in df_limpio.columns:
            df_limpio[columna] = pd.to_numeric(df_limpio[columna], errors="coerce")
    
    return df_limpio


def resumen_limpieza(nombre_fuente, df_original, df_limpio):
    """
    Genera un resumen comparativo antes y después de la limpieza.
    """
    return {
        "fuente": nombre_fuente,
        "filas_originales": df_original.shape[0],
        "filas_limpias": df_limpio.shape[0],
        "columnas_originales": df_original.shape[1],
        "columnas_limpias": df_limpio.shape[1],
        "duplicados_originales": df_original.duplicated().sum(),
        "duplicados_limpios": df_limpio.duplicated().sum(),
        "nulos_originales": df_original.isnull().sum().sum(),
        "nulos_limpios": df_limpio.isnull().sum().sum()
    }

print("Funciones generales de limpieza creadas correctamente.")

Funciones generales de limpieza creadas correctamente.


### 3.1 Limpieza del dataset Global Cybersecurity Threats

En esta fuente se limpian los nombres de columnas, se eliminan duplicados, se normalizan las columnas de texto y se homologa la columna de país. También se convierten las columnas numéricas principales para asegurar que puedan utilizarse en cálculos posteriores.

Las columnas numéricas más importantes son el año, la pérdida financiera, el número de usuarios afectados y el tiempo de resolución del incidente.

In [67]:
def limpiar_threats(df):
    """
    Limpia el DataFrame de amenazas globales de ciberseguridad.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas a formato snake_case
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Eliminación de registros duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país como posible clave de relación
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Conversión de columnas numéricas
    columnas_numericas = [
        "year",
        "financial_loss_in_million",
        "number_of_affected_users",
        "incident_resolution_time_in_hours"
    ]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Conversión del año a entero nullable para conservar consistencia si existieran valores faltantes
    df_limpio["year"] = df_limpio["year"].astype("Int64")
    
    return df_limpio


df_threats_clean = limpiar_threats(df_threats)

df_threats_clean.head()

,country,year,attack_type,target_industry,financial_loss_in_million,number_of_affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_in_hours
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


### 3.2 Limpieza del dataset Cyber Security Index

En esta fuente se limpian los nombres de columnas, se eliminan duplicados, se normalizan los textos y se homologa el país. Además, se corrige la categoría regional **Asia-Pasific** a **Asia-Pacific**, ya que presenta un problema de escritura.

Los indicadores CEI, GCI, NCSI y DDL se convierten a tipo numérico. Los valores nulos se conservan inicialmente para no alterar el significado de los indicadores, pero se agrega una columna auxiliar que identifica si el registro tiene indicadores incompletos.

In [68]:
def limpiar_cyber_security(df):
    """
    Limpia el DataFrame de índices de ciberseguridad por país.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Eliminación de duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Estandarización de categorías de región
    df_limpio["region"] = df_limpio["region"].replace({
        "Asia-Pasific": "Asia-Pacific"
    })
    
    # Conversión de indicadores a tipo numérico
    columnas_numericas = ["cei", "gci", "ncsi", "ddl"]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Columna auxiliar para identificar registros con indicadores incompletos
    df_limpio["has_missing_security_index"] = df_limpio[columnas_numericas].isnull().any(axis=1)
    
    return df_limpio


df_cyber_security_clean = limpiar_cyber_security(df_cyber_security)

df_cyber_security_clean.head()

,country,region,cei,gci,ncsi,ddl,has_missing_security_index
0,Afghanistan,Asia-Pacific,1.000,5.20,11.69,19.50,False
1,Albania,Europe,0.566,64.32,62.34,48.74,False
2,Algeria,Africa,0.721,33.95,33.77,42.81,False
3,Andorra,Europe,NaN,26.38,NaN,NaN,True
4,Angola,Africa,NaN,12.99,9.09,22.69,True


### 3.3 Limpieza del dataset Internet Users by Country

En esta fuente se limpian nombres de columnas, textos y duplicados. La columna **Country or area** se renombra a **country** para facilitar la relación con las otras fuentes. También se convierten a tipo numérico las columnas de usuarios de internet, población, año y porcentaje de usuarios de internet.

Adicionalmente, se crea una columna auxiliar para identificar registros con datos digitales incompletos.

In [69]:
def limpiar_internet_users(df):
    """
    Limpia el DataFrame de usuarios de internet por país.
    """
    df_limpio = df.copy()
    
    # Corrección de nombres de columnas
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    
    # Renombrar campo de país para homologarlo con las otras fuentes
    if "country_or_area" in df_limpio.columns:
        df_limpio = df_limpio.rename(columns={"country_or_area": "country"})
    
    # Eliminación de duplicados
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    
    # Normalización de textos
    df_limpio = limpiar_columnas_texto(df_limpio)
    
    # Homologación de país
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)
    
    # Conversión de columnas numéricas
    columnas_numericas = [
        "internet_users",
        "population_2021",
        "year",
        "percentage_internet_users"
    ]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    
    # Conversión del año a entero nullable
    df_limpio["year"] = df_limpio["year"].astype("Int64")
    
    # Columna auxiliar para identificar registros incompletos en variables digitales clave
    df_limpio["has_missing_digital_data"] = df_limpio[columnas_numericas].isnull().any(axis=1)
    
    return df_limpio


df_internet_users_clean = limpiar_internet_users(df_internet_users)

df_internet_users_clean.head()

,country,subregion,region,internet_users,population_2021,year,percentage_internet_users,has_missing_digital_data
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3,False
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6,False
2,United States,Northern America,Americas,311300000,336997624,2023,92.4,False
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8,False
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8,False


### 3.4 Validación de la limpieza aplicada

Después de limpiar cada fuente, se realiza una validación general para comparar la cantidad de filas, columnas, duplicados y valores nulos antes y después del proceso. Esta revisión permite confirmar que la limpieza no eliminó información de forma innecesaria y que las transformaciones aplicadas mantienen la estructura esperada de los datos.

In [70]:
# Consolidar los DataFrames limpios en un diccionario
clean_datasets = {
    "Global Cybersecurity Threats": df_threats_clean,
    "Cyber Security Index": df_cyber_security_clean,
    "Internet Users by Country": df_internet_users_clean
}

# Resumen comparativo antes y después de la limpieza
resumenes_limpieza = [
    resumen_limpieza(nombre, datasets[nombre], clean_datasets[nombre])
    for nombre in datasets.keys()
]

df_resumen_limpieza = pd.DataFrame(resumenes_limpieza)
df_resumen_limpieza

,fuente,filas_originales,filas_limpias,columnas_originales,columnas_limpias,duplicados_originales,duplicados_limpios,nulos_originales,nulos_limpios
0,Global Cybersecurity Threats,3000,3000,10,10,0,0,0,0
1,Cyber Security Index,192,192,6,7,0,0,151,151
2,Internet Users by Country,215,215,7,8,0,0,0,0


In [71]:
# Revisión de tipos de datos después de la limpieza
for nombre, df in clean_datasets.items():
    print(f"Tipos de datos después de la limpieza: {nombre}")
    display(pd.DataFrame({
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
        "valores_nulos": df.isnull().sum().values,
        "valores_unicos": df.nunique().values
    }))

Tipos de datos después de la limpieza: Global Cybersecurity Threats


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,10
1,year,Int64,0,10
2,attack_type,str,0,6
3,target_industry,str,0,7
4,financial_loss_in_million,float64,0,2536
5,number_of_affected_users,int64,0,2998
6,attack_source,str,0,4
7,security_vulnerability_type,str,0,4
8,defense_mechanism_used,str,0,5
9,incident_resolution_time_in_hours,int64,0,72


Tipos de datos después de la limpieza: Cyber Security Index


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,192
1,region,str,0,5
2,cei,float64,84,85
3,gci,float64,2,180
4,ncsi,float64,25,69
5,ddl,float64,40,150
6,has_missing_security_index,bool,0,2


Tipos de datos después de la limpieza: Internet Users by Country


,columna,tipo_dato,valores_nulos,valores_unicos
0,country,str,0,215
1,subregion,str,0,22
2,region,str,0,5
3,internet_users,int64,0,215
4,population_2021,int64,0,215
5,year,Int64,0,5
6,percentage_internet_users,float64,0,195
7,has_missing_digital_data,bool,0,1


In [72]:
# Validación de países después de la homologación
paises_threats_clean = set(df_threats_clean["country"].dropna().unique())
paises_cyber_security_clean = set(df_cyber_security_clean["country"].dropna().unique())
paises_internet_users_clean = set(df_internet_users_clean["country"].dropna().unique())

paises_en_todas_las_fuentes = (
    paises_threats_clean
    .intersection(paises_cyber_security_clean)
    .intersection(paises_internet_users_clean)
)

print("Países en Global Cybersecurity Threats:", len(paises_threats_clean))
print("Países en Cyber Security Index:", len(paises_cyber_security_clean))
print("Países en Internet Users by Country:", len(paises_internet_users_clean))
print("Países del dataset de amenazas que también existen en las tres fuentes:", len(paises_en_todas_las_fuentes))
print(sorted(paises_en_todas_las_fuentes))

Países en Global Cybersecurity Threats: 10
Países en Cyber Security Index: 192
Países en Internet Users by Country: 215
Países del dataset de amenazas que también existen en las tres fuentes: 10
['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'United Kingdom', 'United States']


### Hallazgos de la limpieza de datos

En la limpieza del dataset **Global Cybersecurity Threats** se corrigieron los nombres de columnas a formato `snake_case`, se eliminaron posibles duplicados, se normalizaron los textos y se homologaron nombres de países como **USA** y **UK**. Además, se validaron las columnas numéricas relacionadas con año, pérdidas financieras, usuarios afectados y tiempo de resolución.

En el dataset **Cyber Security Index** se aplicó la misma normalización de columnas y textos. También se corrigió la categoría regional **Asia-Pasific** a **Asia-Pacific**. Los indicadores CEI, GCI, NCSI y DDL fueron convertidos a tipo numérico y se creó la columna `has_missing_security_index` para identificar registros con indicadores incompletos.

En el dataset **Internet Users by Country** se renombró la columna `country_or_area` a `country`, permitiendo que el campo de país tenga el mismo nombre que en las demás fuentes. También se normalizaron textos, se convirtieron variables numéricas y se creó la columna `has_missing_digital_data` para marcar registros con información digital incompleta.

Esta fase permite contar con tres DataFrames limpios y más consistentes: `df_threats_clean`, `df_cyber_security_clean` y `df_internet_users_clean`. Estos serán utilizados en las siguientes etapas para seleccionar columnas relevantes, crear nuevas variables, generar identificadores y validar los resultados transformados.